In [1]:
from google.colab import userdata
NVIDIA_API_KEY = userdata.get('NVIDIA_API_KEY')

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
! uv pip install langchain langchain_nvidia_ai_endpoints langchain-community pypdf "unstructured[all-docs]" faiss-gpu-cu12 "nemo-microservices[data-designer]" deepeval

Using Python 3.12.12 environment at: /usr
Audited 8 packages in 691ms


In [4]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA
LLM_MODEL = "nvidia/nvidia-nemotron-nano-9b-v2"
llm = ChatNVIDIA(model=LLM_MODEL, temperature=0.6, top_p=0.95, max_tokens=8192, api_key=NVIDIA_API_KEY)


/tmp/ipython-input-1458240435.py:3: DeprecationWarning: The 'max_tokens' parameter is deprecated and will be removed in a future version. Please use 'max_completion_tokens' instead.
  llm = ChatNVIDIA(model=LLM_MODEL, temperature=0.6, top_p=0.95, max_tokens=8192, api_key=NVIDIA_API_KEY)


In [5]:
llm.invoke("Hi")

AIMessage(content='Hello! How can I assist you today? 😊', additional_kwargs={'reasoning_content': 'Okay, the user just said "Hi". That\'s a greeting. I should respond politely. Let me make sure to acknowledge their greeting and offer help. Maybe say something like "Hello! How can I assist you today?" That\'s friendly and opens the door for them to ask questions. I should keep it simple and welcoming.'}, response_metadata={'role': 'assistant', 'reasoning_content': 'Okay, the user just said "Hi". That\'s a greeting. I should respond politely. Let me make sure to acknowledge their greeting and offer help. Maybe say something like "Hello! How can I assist you today?" That\'s friendly and opens the door for them to ask questions. I should keep it simple and welcoming.', 'content': 'Hello! How can I assist you today? 😊', 'tool_calls': [], 'token_usage': {'prompt_tokens': 14, 'total_tokens': 98, 'completion_tokens': 84, 'prompt_tokens_details': None}, 'finish_reason': 'stop', 'model_name': 'n

In [6]:
!apt-get update
!apt-get install -y poppler-utils tesseract-ocr

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Get:4 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:6 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 3,917 B in 1s (2,716 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1

In [7]:

# UNCOMMENT IF NO EMBEDDINGS IN THE DRIVE

# from langchain_community.document_loaders import UnstructuredPDFLoader

# # Recommended values for the 8,192 token limit of NVIDIA NeMo Retriever
# # Assuming ~4 characters per token, 16,000 chars is roughly 4,000 tokens.
# MAX_CHARS = 16000
# SOFT_MAX = 12000

# loader = UnstructuredPDFLoader(
#     "/content/qps-ugs.pdf",
#     mode="elements",                # Essential for identifying Titles, Tables, and Images
#     strategy="auto",                # Fast for text pages, hi_res only when needed
#     chunking_strategy="by_title",   # Keeps section context together
#     max_characters=MAX_CHARS,       # Hard limit to stay under token window
#     new_after_n_chars=SOFT_MAX,     # Preferred chunk size
#     combine_text_under_n_chars=2000, # Merges tiny fragments into larger chunks
#     overlap=0,  # Not needed for clean semantic chunks
#     pdf_infer_table_structure=True,  # Optional: adds HTML version of tables to metadata
#     # show_progress=True,
# )

In [8]:
# UNCOMMENT IF NO EMBEDDINGS IN THE DRIVE

# Load the structured documents
# docs = loader.load()

# print(f"Total chunks created: {len(docs)}")

In [9]:
# UNCOMMENT IF NO EMBEDDINGS IN THE DRIVE


# # Inspect chunks
# import pandas as pd
# from unstructured.staging.base import elements_to_dicts

# # 1. Convert the list of elements/docs into a list of dictionaries
# # If you are using LangChain's 'docs', you can access the original elements if needed
# # or manually create a list of dicts from the Document objects.
# element_dicts = [
#     {
#         "page_content": doc.page_content,
#         "category": doc.metadata.get("category"),
#         "page_number": doc.metadata.get("page_number"),
#         "filename": doc.metadata.get("filename")
#     }
#     for doc in docs
# ]

# # 2. Create the Pandas DataFrame
# df = pd.DataFrame(element_dicts)

# # 3. View the first few rows
# df.head()

In [10]:
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings

client = NVIDIAEmbeddings(
  model="nvidia/llama-3.2-nemoretriever-1b-vlm-embed-v1",
  api_key=NVIDIA_API_KEY,
  truncate="NONE",
  )



In [11]:
# UNCOMMENT IF NO EMBEDDINGS IN THE DRIVE


# from langchain_community.vectorstores import FAISS

# # This will automatically call client.embed_documents() in optimized batches
# vector_store = FAISS.from_documents(docs, client)



In [12]:
# UNCOMMENT IF NO EMBEDDINGS IN THE DRIVE


# # Save to your Google Drive persistently
# vector_store.save_local("/content/drive/My Drive/my_project/faiss_index_fpga_copilot")
# print("Vector store created and saved to Drive.")

In [13]:
!uv pip install rich

Using Python 3.12.12 environment at: /usr
Audited 1 package in 87ms


In [14]:
from rich import print as rprint

In [15]:
# Loading embeddings from the drive

embeddings_path = "/content/drive/My Drive/my_project/faiss_index_fpga_copilot"

In [16]:
import os
from langchain_community.vectorstores import FAISS

if os.path.exists(embeddings_path):
    # Load the vector store
    # You MUST provide the original embedding client (NVIDIAEmbeddings)
    vector_store = FAISS.load_local(
        embeddings_path,
        client,
        allow_dangerous_deserialization=True
    )
    print("Vector store loaded successfully from Drive.")
else:
    print(f"Error: No index found at {embeddings_path}. Did you save it first?")

Vector store loaded successfully from Drive.


In [17]:
# 1. Define your test question
query = "What is the primary function of the Intel Quartus Prime software?"

# 2. Perform the search
# k=3 tells it to find the top 3 matches
results = vector_store.similarity_search(query, k=3)

# 3. Print the results with metadata
for i, res in enumerate(results):
    rprint(f"--- Result {i+1} ---")
    rprint(f"Page: {res.metadata.get('page_number')}")
    rprint(f"Type: {res.metadata.get('category')}")
    rprint(f"Content: {res.page_content[:300]}...\n")

--- Result 1 ---

Page: 6

Type: CompositeElement

Content: Intel Arria® 10, Arria V, and Arria II

Intel Cyclone® 10 LP, Cyclone IV, and Cyclone V

MAX® series

The Intel Quartus Prime software GUI supports easy design entry, fast design processing, straightforward device 
programming, and integration with other industry- standard EDA tools. The user interfa...

--- Result 2 ---

...

Content: Note:

Intel replaces the following Altera tool names in the Intel Quartus Prime software:

Table 1.

Intel Quartus Prime Tool Name Updates

Altera Name

Intel Name

Qsys

Platform Designer

TimeQuest

Timing Analyzer

EyeQ

Eye Viewer

JNEye

Advanced Link Analyzer

Related Information

Migrating t...

--- Result 3 ---

Page: 1293

Type: CompositeElement

Content: Intel Quartus Prime Standard Edition User Guide: Third-party Synthesis

3

683796 | 2018.09.24

Send Feedback

1. Synopsys Synplify* Support

1.1. About Synplify Support

the Intel® Quartus® Prime software supports use of the Synopsys Synplify software design flows, methodologies, and 
techniques for...

In [18]:
# SETUP RETRIEVER TOOL FOR CHAT
from langchain.tools import tool

@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    """Retrieve information to help answer a query."""
    retrieved_docs = vector_store.similarity_search(query, k=2)
    serialized = "\n\n".join(
        (f"Source: {doc.metadata}\nContent: {doc.page_content}")
        for doc in retrieved_docs
    )
    return serialized, retrieved_docs

In [19]:
# CREATE AGENT and append tool
from langchain.agents import create_agent

tools = [retrieve_context]
# If desired, specify custom instructions
prompt = (
    "You have access to a tool that retrieves context from a FPGA Docs. "
    "Use the tool to help answer user queries."
)
agent = create_agent(llm, tools, system_prompt=prompt)

In [20]:
for event in agent.stream({"messages": [("user", query)]}, stream_mode="values"):
    final_message = event["messages"][-1]

print(final_message.content)

The primary function of the Intel Quartus Prime software is to serve as a comprehensive design platform for developing and optimizing FPGA-based systems. It supports the entire design flow, including project setup, synthesis of VHDL/Verilog/SystemVerilog code, design optimization, debugging, and integration with third-party tools. Its modular Compiler and user-friendly interface streamline FPGA development, enabling efficient creation of high-performance hardware designs.


In [21]:
query = (
    "What is the primary function of the Intel Quartus Prime software?"
)

for msg, metadata in agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="messages",
):
    if msg.content:
        print(msg.content, end="", flush=True)

Source: {'source': '/content/qps-ugs.pdf', 'file_directory': '/content', 'filename': 'qps-ugs.pdf', 'last_modified': '2026-01-07T21:27:21', 'page_number': 6, 'orig_elements': 'eJzVW1tz3Day/isovWxyShgTBEACfnM5To6rsmvtyvGeqmzKBQLgDGMOOeZF8ji7/327AXI0o0ssHUfa2QdVqUkQaPTlQ3ej5+ffTnzt174Z3lfu5Dk5YcpkvMxLqrWQVJiU0UKVJS0LJU0muLSFPTklJ2s/GGcGA9/8dmLbtnNVYwbfB7o223Yc3q98tVwN8CTX6SKBj6bnl5UbVvA4Y/Hxpq2aAb/8+WcmJT5Keb7IfzklO1rkC400Z2rBb6HDeHhw0m/7wa9xJ2fVJ1+fb4z1J/+CF2VV+/eu6rwd2m6LA57Zthlg5yfT28asPT7/uOnpuOwXG1eeBKb74f26dVVZ+SCiNEkzmjCa5G9T9jzNn6cMx23M0r9vxnXhO9wbrjn4T7j9k9ewTk1edF1l/jEmifGEwaYCTd6dEtO4iXj9GmcatpvAyNtqqAPz15WUS2+dcsCCSBQVaaGpSoSnptTKpaIsQEuPqSSQNipFiklJE53xiRZsoW+hw/ijVdK+5H+s+uE1sniL8FOZ26IsBDVMeyp4wmkh8pyW3HtTcGWVVE/gIZIv5L6HzMLmOby4hQ7jj1b40UNebm3dNn7nI+THs9P5IXk9ecpMv7uXq7C0VNKzkirODLiKATwD7VGrM13w3Mi0zJ/AVTK94PuukuuovZSxRXoLHcYfrbb+/OL/Jh31vqtASPd0HeW0FaJQVJYupSLngFvGZhT+tLZS24w96uEyCV8nB8rgIo3Cl5yhq1yn4/ijVcbbFfhGcJ+/jqYbxp6cddUaVNOWw6XpPPnhp9ekHzebtht64k2/Jc731bIhwF63PSUl

In [22]:
from google.colab import userdata
OPENAI_API_KEY=userdata.get('OPENAI')

In [23]:
import os
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [24]:
# USING DEEPEVAL SYNTHESISR TO GENERATE A GOLDEN DATASET TO TEST OUR RAG

from deepeval.synthesizer import Synthesizer

# 1. Get random chunks from your vector_store
# Note: most LangChain vector stores have a '.get()' or can be accessed via docstore
all_docs = vector_store.docstore._dict.values() # Example for local FAISS
# Or simply take the first 10 documents for a quick sample
sample_docs = list(all_docs)[30:32]

# 2. Extract context text
contexts = [[doc.page_content] for doc in sample_docs]

# 3. Generate without any prompt configuration
synthesizer = Synthesizer()
goldens = synthesizer.generate_goldens_from_contexts(
    contexts=contexts,
    include_expected_output=True
)

Output()

In [25]:
golden_pd = synthesizer.to_pandas()

In [26]:
golden_pd

,input,actual_output,expected_output,context,retrieval_context,n_chunks_per_context,context_length,evolutions,context_quality,synthetic_input_quality,source_file
0,How do EPE spreadsheet and Power Analyzer in Q...,None,The Early Power Estimator (EPE) spreadsheet in...,[Intel Quartus Prime Standard Edition User Gui...,None,1,2208,[Concretizing],None,1.0,None
1,How does EPE spreadsheet assist in early power...,None,The Early Power Estimator (EPE) spreadsheet he...,[Intel Quartus Prime Standard Edition User Gui...,None,1,2208,[Constrained],None,1.0,None
2,"How do EPE manual entry, Quartus-generated fil...",None,EPE manual entry allows you to input estimated...,[Figure 24.\n\nEarly Power Estimator (EPE) Spr...,None,1,4222,[Multi-context],None,1.0,None
3,How does Pin Planner enable early FPGA I/O int...,None,Pin Planner enables early FPGA I/O interface p...,[Figure 24.\n\nEarly Power Estimator (EPE) Spr...,None,1,4222,[Concretizing],None,1.0,None


In [27]:
qs = golden_pd['input'].tolist()
qs

['How do EPE spreadsheet and Power Analyzer in Quartus Prime assist in pre- and post-design power estimation?',
 'How does EPE spreadsheet assist in early power budgeting and PCB design, considering thermal requirements?',
 'How do EPE manual entry, Quartus-generated files, and Import Data macro collectively complete power estimation?',
 'How does Pin Planner enable early FPGA I/O interface planning and validation using I/O Assignment Analysis?']

In [28]:
res = []
for q in qs:
    print(f"\n--- Processing Query: {q} ---")

    # Store the final result for your 'res' list
    final_output = None

    for event in agent.stream({"messages": [("user", q)]}, stream_mode="values"):
        final_output = event["messages"][-1]
        final_output.pretty_print()

    res.append(final_output.content)


--- Processing Query: How do EPE spreadsheet and Power Analyzer in Quartus Prime assist in pre- and post-design power estimation? ---
================================ Human Message =================================

How do EPE spreadsheet and Power Analyzer in Quartus Prime assist in pre- and post-design power estimation?
================================== Ai Message ==================================
Tool Calls:
  retrieve_context (5dUSvyrMT)
 Call ID: 5dUSvyrMT
  Args:
    query: EPE spreadsheet and Power Analyzer in Quartus Prime for pre- and post-design power estimation
================================= Tool Message =================================
Name: retrieve_context

Source: {'source': '/content/qps-ugs.pdf', 'file_directory': '/content', 'filename': 'qps-ugs.pdf', 'last_modified': '2026-01-07T21:27:21', 'page_number': 1703, 'orig_elements': 'eJzVXF2T28ax/SuofbKrNMx8f+jNZcu5qvKNFWlT98E3pZrPJSokwQCgNusk//32DACaS9IJ5RtsQaUH7QEGJNhnuvt0zwA//f0ubuI27vqPdbh7Xd2pICK10SGZhEI8eIZcIg

In [29]:
(res)

['The **EPE (Early Power Estimator) spreadsheet** and **Intel Quartus Prime Power Analyzer** serve complementary roles in power estimation across different design stages:\n\n### **Pre-Design (Early Estimation) - EPE Spreadsheet**\n1. **Manual Input or Auto-Generated Data**:  \n   - Designers can manually enter device resources (e.g., LUTs, FFs), clock frequencies, toggle rates, and environmental conditions.  \n   - Alternatively, Quartus Prime can generate an EPE file (`.txt`/`.csv`) with resource estimates from a partial or incomplete design.  \n2. **Purpose**:  \n   - Provides a **rough estimate** of power consumption early in the design cycle.  \n   - Helps identify power-critical areas before detailed implementation.  \n3. **Accuracy**:  \n   - Moderate accuracy due to reliance on estimates (e.g., assumed toggle rates or resource usage).  \n\n---\n\n### **Post-Design (Accurate Estimation) - Power Analyzer**\n1. **Post-Fit Analysis**:  \n   - Uses actual placement and routing result

In [30]:
(res[0])

'The **EPE (Early Power Estimator) spreadsheet** and **Intel Quartus Prime Power Analyzer** serve complementary roles in power estimation across different design stages:\n\n### **Pre-Design (Early Estimation) - EPE Spreadsheet**\n1. **Manual Input or Auto-Generated Data**:  \n   - Designers can manually enter device resources (e.g., LUTs, FFs), clock frequencies, toggle rates, and environmental conditions.  \n   - Alternatively, Quartus Prime can generate an EPE file (`.txt`/`.csv`) with resource estimates from a partial or incomplete design.  \n2. **Purpose**:  \n   - Provides a **rough estimate** of power consumption early in the design cycle.  \n   - Helps identify power-critical areas before detailed implementation.  \n3. **Accuracy**:  \n   - Moderate accuracy due to reliance on estimates (e.g., assumed toggle rates or resource usage).  \n\n---\n\n### **Post-Design (Accurate Estimation) - Power Analyzer**\n1. **Post-Fit Analysis**:  \n   - Uses actual placement and routing results

In [31]:
eval_pd = golden_pd.copy()
eval_pd['actual_output'] = res
eval_pd

,input,actual_output,expected_output,context,retrieval_context,n_chunks_per_context,context_length,evolutions,context_quality,synthetic_input_quality,source_file
0,How do EPE spreadsheet and Power Analyzer in Q...,The **EPE (Early Power Estimator) spreadsheet*...,The Early Power Estimator (EPE) spreadsheet in...,[Intel Quartus Prime Standard Edition User Gui...,None,1,2208,[Concretizing],None,1.0,None
1,How does EPE spreadsheet assist in early power...,The EPE (Early Power Estimator) spreadsheet as...,The Early Power Estimator (EPE) spreadsheet he...,[Intel Quartus Prime Standard Edition User Gui...,None,1,2208,[Constrained],None,1.0,None
2,"How do EPE manual entry, Quartus-generated fil...","The EPE (Early Power Estimator) spreadsheet, Q...",EPE manual entry allows you to input estimated...,[Figure 24.\n\nEarly Power Estimator (EPE) Spr...,None,1,4222,[Multi-context],None,1.0,None
3,How does Pin Planner enable early FPGA I/O int...,Pin Planner enables early FPGA I/O interface p...,Pin Planner enables early FPGA I/O interface p...,[Figure 24.\n\nEarly Power Estimator (EPE) Spr...,None,1,4222,[Concretizing],None,1.0,None


In [32]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval import evaluate

# 1. Define the Similarity/Correctness metric
similarity_metric = GEval(
    name="Answer Correctness",
    criteria="Determine if the 'actual output' is factually correct and semantically similar to the 'expected output'.",
    evaluation_params=[
        LLMTestCaseParams.ACTUAL_OUTPUT,
        LLMTestCaseParams.EXPECTED_OUTPUT
    ],
    threshold=0.7
)

# 2. Create test cases from your DataFrame
test_cases = []
for _, row in eval_pd.iterrows():
    test_case = LLMTestCase(
        input=row['input'],
        actual_output=row['actual_output'],
        expected_output=row['expected_output']
    )
    test_cases.append(test_case)

# 3. Run the evaluation
eval_res = evaluate(test_cases, [similarity_metric])

✨ You're running DeepEval's latest Answer Correctness [GEval] Metric! (using gpt-4.1, strict=False, 
async_mode=True)...

Output()

INFO:deepeval.evaluate.execute:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute:in _a_execute_llm_test_cases




Metrics Summary

  - ✅ Answer Correctness [GEval] (score: 0.9851952798209368, threshold: 0.7, strict: False, evaluation model: gpt-4.1, reason: The actual output is factually correct and semantically aligned with the expected output. It accurately describes the roles of EPE manual entry, Quartus-generated files, and the Import Data macro, matching the expected workflow and details. The explanation is more detailed but does not introduce inaccuracies or semantic deviations, and it covers all key aspects required by the evaluation steps., error: None)

For test case:

  - input: How do EPE manual entry, Quartus-generated files, and Import Data macro collectively complete power estimation?
  - actual output: The EPE (Early Power Estimator) spreadsheet, Quartus-generated files, and the Import Data macro work together in a streamlined workflow for power estimation in FPGA designs:

1. **EPE Manual Entry**:  
   - Used when no existing design exists. Designers manually input parameters lik

⚠ WARNING: No hyperparameters logged.
» ]8;id=678484;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 5.92s | token cost: 0.012556000000000001 USD)
» Test Results (4 total tests):
   » Pass Rate: 100.0% | Passed: 4 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

In [33]:
import pandas as pd

flattened_results = []


for result in eval_res.test_results:
    row = {
        "Test Case": result.name,
        "Input": result.input,
        "Actual Output": result.actual_output,
        "Expected Output": result.expected_output,
        "Success": result.success
    }

    for metric in result.metrics_data:
        row[f"{metric.name} Score"] = metric.score
        row[f"{metric.name} Reason"] = metric.reason

    flattened_results.append(row)

df_results = pd.DataFrame(flattened_results)
df_results

,Test Case,Input,Actual Output,Expected Output,Success,Answer Correctness [GEval] Score,Answer Correctness [GEval] Reason
0,test_case_2,"How do EPE manual entry, Quartus-generated fil...","The EPE (Early Power Estimator) spreadsheet, Q...",EPE manual entry allows you to input estimated...,True,0.985195,The actual output is factually correct and sem...
1,test_case_1,How does EPE spreadsheet assist in early power...,The EPE (Early Power Estimator) spreadsheet as...,The Early Power Estimator (EPE) spreadsheet he...,True,0.922270,The actual output is factually correct and sem...
2,test_case_0,How do EPE spreadsheet and Power Analyzer in Q...,The **EPE (Early Power Estimator) spreadsheet*...,The Early Power Estimator (EPE) spreadsheet in...,True,0.922270,The actual output is factually correct and sem...
3,test_case_3,How does Pin Planner enable early FPGA I/O int...,Pin Planner enables early FPGA I/O interface p...,Pin Planner enables early FPGA I/O interface p...,True,0.799101,The actual output accurately describes Pin Pla...
